<a href="https://colab.research.google.com/github/NayeonKim925/20252R0136DATA30400/blob/main/baseline_keyword_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Baseline Keyword Matching Classifier

This Jupyter notebook implements a simple baseline for the hierarchical product classification task.

The task is to assign each product description to one or more leaf classes from a taxonomy of 531 categories. Since the provided training data does not include labels, this baseline uses **keyword matching** rather than supervised learning.

Each class has a set of associated keywords (see `class_related_keywords.txt`). For each product description, the notebook counts the occurrences of these keywords and selects the classes with the highest counts. If no keywords are matched, random classes are selected as a fall‐back.

**Files** used in this notebook:

- `classes.txt`: maps class IDs to class names.
- `class_related_keywords.txt`: maps class names to comma-separated keywords.
- `test_corpus.txt`: contains product IDs and descriptions (tab separated).

The notebook writes the predictions into a CSV file named `submission.csv` with two columns: `id` and `label`. The `label` column contains comma-separated class IDs for each product.

This baseline is intended as a starting point. It can be extended with more sophisticated text processing, graph-based models, or pretrained language models to achieve higher accuracy.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [17]:
import os
import csv
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer

# === Configuration ===
NUM_CLASSES = 531
MIN_LABELS = 2
MAX_LABELS = 3
SEED = 42

# 파일 경로 (사용자 환경에 맞게 수정)
DATA_DIR = "/content/drive/MyDrive/Fall25 BDA/Amazon_products"

CLASS_FILE = os.path.join(DATA_DIR, "classes.txt")
KEYWORD_FILE = os.path.join(DATA_DIR, "class_related_keywords.txt")
HIER_FILE = os.path.join(DATA_DIR, "class_hierarchy.txt")
TRAIN_FILE = os.path.join(DATA_DIR, "train", "train_corpus.txt")
TEST_FILE = os.path.join(DATA_DIR, "test", "test_corpus.txt")
OUTPUT_FILE = os.path.join(DATA_DIR, "submission_final_fixed.csv")

# === Reproducibility ===
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# === Data Loading ===
def load_classes(path):
    id2name, name2id = {}, {}
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                cid, cname = int(parts[0]), parts[1]
                id2name[cid] = cname
                name2id[cname] = cid
    return id2name, name2id

def load_hierarchy(path):
    parents = defaultdict(list)
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            p, c = map(int, line.strip().split('\t'))
            parents[c].append(p)
    return parents

def load_keywords(path, name2id):
    class_keywords = {}
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if ':' in line:
                cname, kw_str = line.strip().split(':', 1)
                keywords = [k.strip() for k in kw_str.split(',')]
                if cname in name2id:
                    class_keywords[name2id[cname]] = keywords
    return class_keywords

def load_corpus(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t', 1)
            if len(parts) == 2:
                did, text = parts
                data.append((int(did), text))
    # 학습/테스트 시에는 정렬해서 처리
    data.sort(key=lambda x: x[0])
    return data

# === Silver Label Generation ===
def generate_silver_labels(corpus, class_keywords, parents_map):
    labels = np.zeros((len(corpus), NUM_CLASSES), dtype=np.float32)

    for idx, (_, text) in enumerate(corpus):
        text_lower = text.lower()
        matched_classes = set()

        for cid, keywords in class_keywords.items():
            for kw in keywords:
                if kw in text_lower:
                    matched_classes.add(cid)
                    break

        queue = list(matched_classes)
        visited = set(queue)
        while queue:
            curr = queue.pop(0)
            if curr in parents_map:
                for p in parents_map[curr]:
                    if p not in visited:
                        visited.add(p)
                        queue.append(p)

        final_labels = list(visited)

        if len(final_labels) < MIN_LABELS:
            final_labels.append(0)
            while len(final_labels) < MIN_LABELS:
                rand_class = random.randint(1, NUM_CLASSES - 1)
                if rand_class not in final_labels:
                    final_labels.append(rand_class)

        if len(final_labels) > MAX_LABELS:
            final_labels = final_labels[:MAX_LABELS]

        for cid in final_labels:
            labels[idx, cid] = 1.0

    return labels

# === Dataset & Model ===
class TextDataset(Dataset):
    def __init__(self, features, labels=None):
        self.features = torch.FloatTensor(features)
        self.labels = torch.FloatTensor(labels) if labels is not None else None

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        if self.labels is not None:
            return self.features[idx], self.labels[idx]
        return self.features[idx]

class SimpleClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(SimpleClassifier, self).__init__()
        self.layer = nn.Sequential(
            nn.Linear(input_dim, 2048),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(2048, 1024),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(1024, num_classes)
        )

    def forward(self, x):
        return self.layer(x)

# === Main Pipeline ===
print("Loading data...")
id2name, name2id = load_classes(CLASS_FILE)
parents_map = load_hierarchy(HIER_FILE)
class_keywords = load_keywords(KEYWORD_FILE, name2id)

train_data = load_corpus(TRAIN_FILE)
test_data = load_corpus(TEST_FILE)

# TF-IDF
print("Vectorizing...")
all_texts = [t for _, t in train_data] + [t for _, t in test_data]
vectorizer = TfidfVectorizer(max_features=10000, stop_words='english', min_df=5)
tfidf_matrix = vectorizer.fit_transform(all_texts)

train_features = tfidf_matrix[:len(train_data)].toarray()
test_features = tfidf_matrix[len(train_data):].toarray()

# Silver Labels
print("Generating labels...")
train_labels = generate_silver_labels(train_data, class_keywords, parents_map)

# Training
print("Training...")
train_dataset = TextDataset(train_features, train_labels)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

model = SimpleClassifier(train_features.shape[1], NUM_CLASSES).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

model.train()
for epoch in range(7):
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        out = model(bx)
        loss = criterion(out, by)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1} done.")

# Inference
print("Inference...")
test_dataset = TextDataset(test_features)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

model.eval()
all_preds = []
with torch.no_grad():
    for bx in test_loader:
        bx = bx.to(device)
        logits = model(bx)
        probs = torch.sigmoid(logits)
        _, top_indices = torch.topk(probs, k=MAX_LABELS, dim=1)
        all_preds.extend(top_indices.cpu().numpy())

# 결과 매핑 준비
id_to_pred = {}
for i, (did, _) in enumerate(test_data):
    id_to_pred[did] = all_preds[i]

# === Submission 저장 ===

print(f"Saving submission to {OUTPUT_FILE}...")
raw_pids = []
with open(TEST_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split('\t', 1)
        if len(parts) == 2:
            raw_pids.append(int(parts[0]))

with open(OUTPUT_FILE, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["id", "labels"])

    for pid in raw_pids:
        if pid in id_to_pred:
            preds = id_to_pred[pid]
            pred_str = ",".join(map(str, preds))
            writer.writerow([pid, pred_str])
        else:
            writer.writerow([pid, "0,1"])

print("Done! Submission file created.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Error: /content/drive/MyDrive/Amazon_products 경로에 데이터가 없습니다. 경로를 확인해주세요!
Running on device: cpu (Speed Mode)


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Amazon_products/classes.txt'